In [ ]:
import import_ipynb
import Metric_Calculation
import pandas as pd
import math
import numpy as np
from numba import njit

In [ ]:
@njit
def _compute_Pr(P, E, x1):
    #compute Pn, Ps, En, Es, Perc, Pr
    Pr = np.zeros_like(P)
    S = 0.3*x1
    for i in range(len(P)):
        Pn = Ps = En = Es = 0
        if P[i]>E[i]:
            Pn = P[i]-E[i]
            Ps = x1*(1-(S/x1)**2)*math.tanh(Pn/x1)/((1+S/x1*math.tanh(Pn/x1)))
            S = S+Ps
        else:
            En = E[i]-P[i]
            Es = S*(2-(S/x1))*math.tanh(En/x1)/(1+(1-S/x1)*math.tanh(En/x1))
            S = S-Es
        if S<0:
            S=0
        if S>x1:
            S=x1
        Perc = S*(1-(1+(4*S/(9*x1))**4)**(-1/4))
        Pr[i] = Perc+(Pn-Ps)
        S = S-Perc
    return Pr

In [ ]:
@njit
def _compute_UH(x4):
    n = math.ceil(x4)
    m = math.ceil(2*x4)
    SH1 = np.zeros(n+1)
    SH2 = np.zeros(m+1)
    UH1 = np.zeros(n+1)
    UH2 = np.zeros(m+1)
    for t in range(n+1):
        if t<=0:
            SH1[t] = 0
        elif t>0 and t<x4:
            SH1[t] = (t/x4)**(5/2)
        else:
            SH1[t] = 1
        if t==0:
            a = SH1[t]
            # print(f"SH1[{t}] = {SH1[t]}")
        else:
            UH1[t-1] = SH1[t] - a
            a = SH1[t]
            # print(f"UH1[{t-1}] = {UH1[t-1]}, SH1[{t}] = {SH1[t]}")
    for t in range(m+1):
        if t<=0:
            SH2[t] = 0
        elif t>0 and t<x4:
            SH2[t] = 1/2*(t/x4)**(5/2)
        elif t>x4 and t<2*x4:
            SH2[t] = 1-1/2*(2-t/x4)**(5/2) 
        else:
            SH2[t] = 1
        if t==0:
            a = SH2[t]
            # print(f"SH2[{t}] = {SH2[t]}")
        else:
            UH2[t-1] = SH2[t] - a
            a = SH2[t]
            # print(f"UH2[{t-1}] = {UH2[t-1]}, SH2[{t}] = {SH2[t]}")
    return UH1, UH2

In [ ]:
# @njit-compiled routing loop (np.convolve isn't supported in nopython mode, so it
# stays outside; the per-step store update is the hot path and is jitted here).
@njit
def _route(Q9, Q1, x2, x3, R):
    n = len(Q9)
    Q = np.zeros(n)
    for i in range(n):
        F = x2*(R/x3)**(7/2)
        if (R+Q9[i]+F)<0:
            R = 0.0
        else:
            R = R+Q9[i]+F
        Qr = R*(1-(1+(R/x3)**4)**(-1/4))
        R = R - Qr
        if Q1[i]+F<0:
            Qd = 0.0
        else:
            Qd = Q1[i]+F
        Q[i] = Qr+Qd
    return Q

def compute_Q(P, E, x):
    P = np.asarray(P, dtype=np.float64)     # njit wants float64 arrays
    E = np.asarray(E, dtype=np.float64)
    [x1, x2, x3, x4] = x
    R = 0.5*x3
    #compute Pn, Ps, En, Es, Perc, Pr
    Pr = _compute_Pr(P, E, x1)
    #compute Uh1 & UH2
    UH1, UH2 = _compute_UH(x4)
    #Compute Q
    Q9 = 0.9 * np.convolve(Pr, UH1)[:len(Pr)]
    Q1 = 0.1 * np.convolve(Pr, UH2)[:len(Pr)]
    return _route(Q9, Q1, x2, x3, R)

In [ ]:
csv_path = r"D:\Claude\flood_hydrology_modeling\data\processed\prec_PET_sf.csv"
out_path = r"D:\Claude\flood_hydrology_modeling\data\processed\GR4J.csv"
df = pd.read_csv(csv_path)
P = df['prec_AGCD']
E = df['PET_morton']
x1, x2, x3, x4 = [350, 0, 90, 1.7]
x = np.array((x1,x2,x3,x4))
%timeit Q = compute_Q(P, E, x)
Q = compute_Q(P, E, x)
ref = pd.read_csv(out_path)['Q'].to_numpy()
print(np.abs(Q - ref).max())

In [ ]:
print(Q)